In [ ]:
!pip install python-terrier pyterrier-rag pyterrier-alpha scikit-learn -q

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pyterrier as pt

# Load NQ dataset
dataset = pt.get_dataset('rag:nq')
train_topics = dataset.get_topics('train')
train_answers = dataset.get_answers('train')
test_topics = dataset.get_topics('test')
test_answers = dataset.get_answers('test')

print('Train queries:', len(train_topics))
print('Test queries:', len(test_topics))
print(train_topics[:3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train queries: 79168
Test queries: 3610
       qid                                        query
0  train_0  total number of death row inmates in the us
1  train_1   big little lies season 2 how many episodes
2  train_2         who sang waiting for a girl like you


In [ ]:
# Download Wikipedia BM25 index (fast on Colab)
print('Downloading Wikipedia index...')
sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
bm25 = pt.rewrite.tokenise() >> sparse_index.bm25(include_fields=['docno', 'text', 'title']) >> pt.rewrite.reset()
print('Index ready!')

https://huggingface.co/datasets/pyterrier/ragwiki-terrier/resolve/main/artifact.tar.lz4:   0%|          | 0.00…

extracting data.direct.bf [1.9 GB]
extracting data.document.fsarrayfile [340.7 MB]
extracting data.inverted.bf [1.5 GB]
extracting data.lexicon.fsomapfile [330.0 MB]
extracting data.lexicon.fsomaphash [1017 B]
extracting data.lexicon.fsomapid [15.3 MB]
extracting data.meta-0.fsomapfile [1.3 GB]
extracting data.meta.idx [160.3 MB]
extracting data.meta.zdata [8.2 GB]
extracting data.properties [4.1 KB]
extracting pt_meta.json [79 B]
terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started (triggered by tokenise) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


13:46:03.568 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading lookup file directly from disk (SLOW) - try index.meta.index-source=fileinmem in the index properties file. 160.3 MiB of memory would be required.
13:46:03.605 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 8.2 GiB of memory would be required.
Index ready!


In [ ]:
import pandas as pd

# Use small subset for training (full 79k would take hours)
train_sample = train_topics.head(500)
answers_sample = train_answers[train_answers['qid'].isin(train_sample['qid'])]

print('Retrieving docs for 500 train queries...')
train_retrieved = bm25 % 10 >> pt.transformer.IdentityTransformer()
train_results = train_retrieved.transform(train_sample)
print('Done! Shape:', train_results.shape)
print(train_results[:3])

Retrieving docs for 500 train queries...
Done! Shape: (5000, 8)
       qid     docid     docno  \
0  train_0   4830496   4830496   
1  train_0  12922184  12922184   
2  train_0    279431    279431   

                                                text  \
0  punishable by death penalty. But perhaps the b...   
1  West Livingston; female death row inmates are ...   
2  row inmates previously at the Tucker Unit, wer...   

                              title  rank      score  \
0    "Presidency of Joseph Estrada"     0  43.515003   
1     "Capital punishment in Texas"     1  43.209411   
2  "Capital punishment in Arkansas"     2  42.717275   

                                         query  
0  total number of death row inmates in the us  
1  total number of death row inmates in the us  
2  total number of death row inmates in the us  


In [ ]:
!pip install -q python-terrier pyterrier-rag pyterrier-alpha scikit-learn
!pip install -q git+https://github.com/pandeylakshya207-max/pyterrier_rag.git@feature/aicl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip show pyterrier-rag | grep Location
!ls $(python -c "import pyterrier_rag; import os; print(os.path.dirname(pyterrier_rag.__file__))")

Location: /usr/local/lib/python3.12/dist-packages
backend       _measures.py  prompt	    readers
_datasets.py  measures.py   pt_docs	    search_o1.py
frameworks    model.py	    __pycache__     search_r1.py
__init__.py   _optional.py  r1_searcher.py  _util.py


In [ ]:
!curl -o /usr/local/lib/python3.12/dist-packages/pyterrier_rag/aicl.py https://raw.githubusercontent.com/pandeylakshya207-max/pyterrier_rag/feature/aicl/pyterrier_rag/aicl.py
from pyterrier_rag.aicl import AICLContextSelector
print('Import works!')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8399  100  8399    0     0  21416      0 --:--:-- --:--:-- --:--:-- 21371
Import works!


In [ ]:
import pyterrier as pt
import pandas as pd
from pyterrier_rag.aicl import AICLContextSelector

# Load NQ dataset
dataset = pt.get_dataset('rag:nq')
train_topics = dataset.get_topics('train').head(500)
train_answers = dataset.get_answers('train')
answers_sample = train_answers[train_answers['qid'].isin(train_topics['qid'])]
test_topics = dataset.get_topics('test').head(200)
test_answers = dataset.get_answers('test')
test_answers_sample = test_answers[test_answers['qid'].isin(test_topics['qid'])]
print('Dataset loaded.')

# Load index
sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
bm25 = pt.rewrite.tokenise() >> sparse_index.bm25(include_fields=['docno','text','title']) >> pt.rewrite.reset()
print('Index loaded.')

# Retrieve docs
print('Retrieving train docs...')
train_results = (bm25 % 10).transform(train_topics)
print('Retrieving test docs...')
test_results = (bm25 % 10).transform(test_topics)
print('Retrieval done. Train shape:', train_results.shape)

# Build labels
print('Building labels...')
labels = AICLContextSelector.build_labels_from_results(
    train_results, answers_sample, answer_col='answers', k_max=10
)
print('Labels built:', len(labels), 'queries')
print('Example:', labels[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset loaded.


Java started (triggered by tokenise) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


14:29:33.328 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading lookup file directly from disk (SLOW) - try index.meta.index-source=fileinmem in the index properties file. 160.3 MiB of memory would be required.
14:29:33.407 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 8.2 GiB of memory would be required.
Index loaded.
Retrieving train docs...
Retrieving test docs...
Retrieval done. Train shape: (5000, 8)
Building labels...


KeyError: 'answers'

In [ ]:
print(train_answers.columns.tolist())
print(train_answers[:3])

['qid', 'gold_answer']
       qid gold_answer
0  train_0       2,718
1  train_1       seven
2  train_2   Foreigner


In [ ]:
!pip install -q python-terrier pyterrier-rag pyterrier-alpha scikit-learn
!curl -s -o /usr/local/lib/python3.12/dist-packages/pyterrier_rag/aicl.py https://raw.githubusercontent.com/pandeylakshya207-max/pyterrier_rag/feature/aicl/pyterrier_rag/aicl.py

import pyterrier as pt
import pandas as pd
from pyterrier_rag.aicl import AICLContextSelector
print('All imports work!')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Load NQ dataset
dataset = pt.get_dataset('rag:nq')
train_topics = dataset.get_topics('train').head(500)
train_answers = dataset.get_answers('train')
answers_sample = train_answers[train_answers['qid'].isin(train_topics['qid'])]
test_topics = dataset.get_topics('test').head(200)
test_answers = dataset.get_answers('test')
test_answers_sample = test_answers[test_answers['qid'].isin(test_topics['qid'])]
print('Dataset loaded. Train:', len(train_topics), 'Test:', len(test_topics))

# Load index
print('Loading index...')
sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
bm25 = pt.rewrite.tokenise() >> sparse_index.bm25(include_fields=['docno','text','title']) >> pt.rewrite.reset()
print('Index ready.')

# Retrieve
print('Retrieving train docs...')
train_results = (bm25 % 10).transform(train_topics)
print('Retrieving test docs...')
test_results = (bm25 % 10).transform(test_topics)
print('Done. Train shape:', train_results.shape, 'Test shape:', test_results.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset loaded. Train: 500 Test: 200
Loading index...


https://huggingface.co/datasets/pyterrier/ragwiki-terrier/resolve/main/artifact.tar.lz4:   0%|          | 0.00…

extracting data.direct.bf [1.9 GB]
extracting data.document.fsarrayfile [340.7 MB]
extracting data.inverted.bf [1.5 GB]
extracting data.lexicon.fsomapfile [330.0 MB]
extracting data.lexicon.fsomaphash [1017 B]
extracting data.lexicon.fsomapid [15.3 MB]
extracting data.meta-0.fsomapfile [1.3 GB]
extracting data.meta.idx [160.3 MB]
extracting data.meta.zdata [8.2 GB]
extracting data.properties [4.1 KB]
extracting pt_meta.json [79 B]
terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started (triggered by tokenise) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


17:15:29.037 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading lookup file directly from disk (SLOW) - try index.meta.index-source=fileinmem in the index properties file. 160.3 MiB of memory would be required.
17:15:29.072 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 8.2 GiB of memory would be required.
Index ready.
Retrieving train docs...
Retrieving test docs...
Done. Train shape: (5000, 8) Test shape: (2000, 8)


In [ ]:
# Build training labels
print('Building labels...')
labels = AICLContextSelector.build_labels_from_results(
    train_results, answers_sample, answer_col='gold_answer', k_max=10
)
print('Labels built:', len(labels))

# Fit AICL
print('Fitting AICL...')
aicl = AICLContextSelector(k_max=10)
aicl.fit(train_results, labels)
print('AICL fitted!')

# Apply AICL on test set
print('Applying AICL on test queries...')
test_filtered = aicl.transform(test_results)

# Compare context sizes
avg_before = len(test_results) / len(test_topics)
avg_after = len(test_filtered) / len(test_topics)
print(f'\nAverage docs per query BEFORE AICL: {avg_before:.1f}')
print(f'Average docs per query AFTER AICL:  {avg_after:.1f}')
print(f'Context reduction: {100*(1 - avg_after/avg_before):.1f}%')

# Show predicted k distribution
import pandas as pd
k_preds = aicl.predict_k(test_results)
k_values = list(k_preds.values())
print(f'\nPredicted k distribution:')
print(pd.Series(k_values).value_counts().sort_index())

Building labels...
Labels built: 500
Fitting AICL...
AICL fitted!
Applying AICL on test queries...

Average docs per query BEFORE AICL: 10.0
Average docs per query AFTER AICL:  10.0
Context reduction: 0.0%

Predicted k distribution:
10    200
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

# Use Random Forest instead - better for this task
aicl2 = AICLContextSelector(
    k_max=10,
    classifier=MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
)
aicl2.fit(train_results, labels)
print('AICL fitted with Random Forest!')

# Apply on test
test_filtered2 = aicl2.transform(test_results)
avg_after2 = len(test_filtered2) / len(test_topics)
print(f'Average docs per query BEFORE: 10.0')
print(f'Average docs per query AFTER:  {avg_after2:.1f}')
print(f'Context reduction: {100*(1 - avg_after2/10):.1f}%')

k_preds2 = aicl2.predict_k(test_results)
k_values2 = list(k_preds2.values())
print(f'\nPredicted k distribution:')
print(pd.Series(k_values2).value_counts().sort_index())

# Show label distribution
import numpy as np
label_arr = np.array(labels)
print(f'\nLabel stats (how often answer found at each k):')
for k in range(10):
    print(f'  k={k+1}: {label_arr[:,k].mean()*100:.1f}% queries have answer')

AICL fitted with Random Forest!
Average docs per query BEFORE: 10.0
Average docs per query AFTER:  9.7
Context reduction: 2.7%

Predicted k distribution:
4       1
5       1
6       3
7       2
8       6
9      13
10    174
Name: count, dtype: int64

Label stats (how often answer found at each k):
  k=1: 21.2% queries have answer
  k=2: 28.8% queries have answer
  k=3: 35.0% queries have answer
  k=4: 38.2% queries have answer
  k=5: 43.2% queries have answer
  k=6: 45.0% queries have answer
  k=7: 46.8% queries have answer
  k=8: 49.6% queries have answer
  k=9: 50.4% queries have answer
  k=10: 51.2% queries have answer


In [ ]:
print("="*50)
print("AICL RESULTS SUMMARY ON NQ DATASET")
print("="*50)
print(f"Training queries: 500")
print(f"Test queries: 200")
print(f"Max k: 10")
print()
print(f"BEFORE AICL: fixed k=10 docs per query")
print(f"AFTER AICL:  avg {avg_after2:.1f} docs per query")
print(f"Context reduction: {100*(1 - avg_after2/10):.1f}%")
print()
print("Answer found in top-k docs (train set):")
for k in range(10):
    print(f"  k={k+1}: {label_arr[:,k].mean()*100:.1f}%")
print()
print("AICL successfully adapts context size per query!")
print("Queries needing fewer docs get smaller context,")
print("reducing noise for the LLM reader.")

AICL RESULTS SUMMARY ON NQ DATASET
Training queries: 500
Test queries: 200
Max k: 10

BEFORE AICL: fixed k=10 docs per query
AFTER AICL:  avg 9.7 docs per query
Context reduction: 2.7%

Answer found in top-k docs (train set):
  k=1: 21.2%
  k=2: 28.8%
  k=3: 35.0%
  k=4: 38.2%
  k=5: 43.2%
  k=6: 45.0%
  k=7: 46.8%
  k=8: 49.6%
  k=9: 50.4%
  k=10: 51.2%

AICL successfully adapts context size per query!
Queries needing fewer docs get smaller context,
reducing noise for the LLM reader.
